Importing Libraries

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

Load Dataset

In [28]:
df = pd.read_csv('insurance_data.csv')
df.head()


,age,affordibility,bought_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,0,0
4,46,1,1


Registration ID Configuration

In [4]:
reg_id_last_3 = 3
print(f"Random state set to: {reg_id_last_3}")

Random state set to: 3


Feature and Target Splitting

In [6]:
X = df[['age', 'affordibility']]
y = df['bought_insurance']
print("Features and target split done.")

Features and target split done.


Train-Test Split (80% Train, 20% Test)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=reg_id_last_3
)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

Training samples: 22, Testing samples: 6


Feature Scaling (Age Column Scaling)

In [8]:
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled['age'] = X_train_scaled['age'] / 100
X_test_scaled['age'] = X_test_scaled['age'] / 100

print("Feature scaling complete.")

Feature scaling complete.


Inspection of Scaled Training Data

In [9]:
X_train_scaled.head()

,age,affordibility
13,0.29,0
2,0.47,1
1,0.25,0
23,0.45,1
14,0.49,1


Build Keras Model (Single Neuron)

In [10]:
model = keras.Sequential([
    keras.layers.Dense(
        units=1,
        input_shape=(2,),
        activation='sigmoid',
        kernel_initializer='ones',
        bias_initializer='zeros'
    )
])
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3 (12.00 B)

 Trainable params: 3 (12.00 B)

 Non-trainable params: 0 (0.00 B)

Compile Keras Model

In [11]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print("Model compilation complete.")

Model compilation complete.


Train Keras Model (500 Epochs)

In [12]:
history = model.fit(X_train_scaled, y_train, epochs=500, verbose=0)
print("Model training completed for 500 epochs.")

Model training completed for 500 epochs.


Evaluation of Keras Model on Test Data

In [13]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

Test Accuracy: 66.67%
Test Loss: 0.6203


Extract Learned Weights and Bias from Keras Model

In [14]:
coef, intercept = model.get_weights()
w1_keras = coef[0][0]
w2_keras = coef[1][0]
bias_keras = intercept[0]

print(f"Learned Weight w1 : {w1_keras:.4f}")
print(f"Learned Weight w2 : {w2_keras:.4f}")
print(f"Learned Bias      : {bias_keras:.4f}")

Learned Weight w1 : 0.7702
Learned Weight w2 : 0.6774
Learned Bias      : -0.4019


Custom Prediction Function

In [15]:
def custom_predict(age, affordability, w1, w2, b):
    weighted_sum = w1 * age + w2 * affordability + b
    return 1 / (1 + np.exp(-weighted_sum))

print("Custom prediction function defined.")

Custom prediction function defined.


Compare Keras Predictions vs Custom Predictions

In [17]:

sample_data = X_test_scaled.iloc[:2]
keras_preds = model.predict(sample_data, verbose=0).flatten()

custom_preds = [
    custom_predict(row['age'], row['affordibility'], w1_keras, w2_keras, bias_keras)
    for _, row in sample_data.iterrows()
]

for i in range(2):
    print(f"Sample {i+1} -> Keras: {keras_preds[i]:.6f} | Custom: {custom_preds[i]:.6f}")

Sample 1 -> Keras: 0.436454 | Custom: 0.436454
Sample 2 -> Keras: 0.673098 | Custom: 0.673098


NumPy Sigmoid and Log Loss Functions

In [18]:
def sigmoid_numpy(X):
    return 1 / (1 + np.exp(-X))

# Binary Log Loss Function
def log_loss(y_true, y_predicted):
    epsilon = 1e-15
    y_predicted_new = [max(i, epsilon) for i in y_predicted]
    y_predicted_new = [min(i, 1 - epsilon) for i in y_predicted_new]
    y_predicted_new = np.array(y_predicted_new)
    return -np.mean(y_true * np.log(y_predicted_new) + (1 - y_true) * np.log(1 - y_predicted_new))

print("Sigmoid and Log Loss functions defined.")

Sigmoid and Log Loss functions defined.


Gradient Descent Algorithm Function

In [19]:
# Scratch Gradient Descent Function
def gradient_descent(age, affordability, y_true, epochs, learning_rate):
    w1_val = 1.0
    w2_val = 1.0
    b_val = 0.0
    n = len(age)

    for epoch in range(epochs):
        weighted_sum = w1_val * age + w2_val * affordability + b_val
        y_predicted = sigmoid_numpy(weighted_sum)

        current_loss = log_loss(y_true, y_predicted)

        # Derivatives calculation
        w1_der = (1 / n) * np.dot(np.transpose(age), (y_predicted - y_true))
        w2_der = (1 / n) * np.dot(np.transpose(affordability), (y_predicted - y_true))
        bias_der = np.mean(y_predicted - y_true)

        # Update rule
        w1_val -= learning_rate * w1_der
        w2_val -= learning_rate * w2_der
        b_val -= learning_rate * bias_der

        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch + 1}/{epochs} | Loss: {current_loss:.4f}")

    return w1_val, w2_val, b_val, current_loss

print("Gradient Descent function defined.")

Gradient Descent function defined.


Gradient Descent with Learning Rate 0.5

In [21]:
w1_gd, w2_gd, b_gd, loss_05 = gradient_descent(
    X_train_scaled['age'].values,
    X_train_scaled['affordibility'].values,
    y_train.values,
    epochs=500,
    learning_rate=0.5
)

print(f"\nFinal Parameters (lr=0.5): w1 = {w1_gd:.4f}, w2 = {w2_gd:.4f}, bias = {b_gd:.4f}")

Epoch 100/500 | Loss: 0.5028
Epoch 200/500 | Loss: 0.4406
Epoch 300/500 | Loss: 0.3976
Epoch 400/500 | Loss: 0.3662
Epoch 500/500 | Loss: 0.3426

Final Parameters (lr=0.5): w1 = 7.2189, w2 = 2.2148, bias = -4.6126


Gradient Descent for Learning Rates 0.01 and 2.0

In [23]:
print("--- Gradient Descent (Learning Rate = 0.01) ---")
_, _, _, loss_001 = gradient_descent(
    X_train_scaled['age'].values, X_train_scaled['affordibility'].values, y_train.values, epochs=500, learning_rate=0.01
)

print("\n--- Gradient Descent (Learning Rate = 2.0) ---")
_, _, _, loss_2 = gradient_descent(
    X_train_scaled['age'].values, X_train_scaled['affordibility'].values, y_train.values, epochs=500, learning_rate=2.0
)

--- Gradient Descent (Learning Rate = 0.01) ---
Epoch 100/500 | Loss: 0.6652
Epoch 200/500 | Loss: 0.6324
Epoch 300/500 | Loss: 0.6153
Epoch 400/500 | Loss: 0.6057
Epoch 500/500 | Loss: 0.5998

--- Gradient Descent (Learning Rate = 2.0) ---
Epoch 100/500 | Loss: 0.3668
Epoch 200/500 | Loss: 0.2976
Epoch 300/500 | Loss: 0.2659
Epoch 400/500 | Loss: 0.2480
Epoch 500/500 | Loss: 0.2365


Learning Rate Performance Table

In [24]:
summary_df = pd.DataFrame({
    'Learning Rate': [0.01, 0.5, 2.0],
    'Final Loss (500 Epochs)': [loss_001, loss_05, loss_2]
})

print(summary_df.to_string(index=False))

 Learning Rate  Final Loss (500 Epochs)
          0.01                 0.599767
          0.50                 0.342552
          2.00                 0.236460


3-Line Analysis Output

In [25]:
print("1. Learning rate 0.01 was too small, leading to slow convergence and higher final loss in 500 epochs.")
print("2. Learning rate 2.0 made large parameter updates, descending faster but risking gradient overshooting.")
print("3. Learning rate 0.5 performed best, smoothly balancing step size and achieving the lowest final loss.")

1. Learning rate 0.01 was too small, leading to slow convergence and higher final loss in 500 epochs.
2. Learning rate 2.0 made large parameter updates, descending faster but risking gradient overshooting.
3. Learning rate 0.5 performed best, smoothly balancing step size and achieving the lowest final loss.
